<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Train_MLFF_ALLEGRO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install wandb
!pip install -q wandb

# install nequip
!pip install -q nequip==0.5.5 torch==1.11  jarvis-tools

# fix colab imports
import site
site.main()

# set to allow anonymous WandB
import os
os.environ["WANDB_ANONYMOUS"] = "must"


In [ ]:
!wget https://figshare.com/ndownloader/files/40357663 -O mlearn.json.zip

In [ ]:
import json,zipfile
mlearn = json.loads(
        zipfile.ZipFile("mlearn.json.zip").read(
            "mlearn.json"
        )
    )

In [ ]:
# install allegro
!git clone --depth 1 https://github.com/mir-group/allegro.git
!pip install allegro/

In [ ]:
import os
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/usnistgov/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -e .

In [ ]:
%%time
import os,glob,sys,yaml,pprint
import zipfile
import json
import pandas as pd
from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms
import numpy as np
from nequip.data import AtomicData, Collater, dataset_from_config, register_fields, AtomicDataDict
from nequip.data.transforms import TypeMapper
import fileinput
import torch
os.chdir('/content')
#torch.cuda.is_available = lambda : False
elements = ["Si"] #["Ni", "Si", "Ge", "Mo", "Cu", "Li"]

with open('allegro/configs/tutorial.yaml','r') as f:
    txt=f.read()

tut = yaml.load(txt, Loader=yaml.Loader)

os.environ["WANDB_ANONYMOUS"] = "must"
cmd = "wandb offline"
os.system(cmd)
#mlearn = data("mlearn")


# def replaceAll(filename,searchExp,replaceExp):
#     with open(filename, "r") as file:
#          filedata = file.read().splitlines()
#     content = []
#     for j in filedata:
#         if searchExp in j:
#            content.append(replaceExp)
#         else:
#             content.append(j)
#     with open(filename, "w") as file:
#          file.write("\n".join(content))
for element in elements:
    os.chdir('/content')
    cmd = "rm -r Si_data"
    os.system(cmd)
    folder = "Si_data"
    if not os.path.exists(folder):
        os.makedirs(folder)
    benchmark_energies = (
        "jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/MLFF/mlearn_"
        + element
        + "_energy.json.zip"
    )
    temp_energies = benchmark_energies.split("/")[-1].split(".zip")[0]
    energies = json.loads(
        zipfile.ZipFile(benchmark_energies).read(temp_energies)
    )
    train_ids = list(energies["train"].keys())
    test_ids = list(energies["test"].keys())

    f = open("Si_data/sitraj.xyz", "w")
    line = ""
    for i in mlearn:
        if i["jid"] in train_ids:
            # print(i)
            atoms = Atoms.from_dict(i["atoms"])
            line += str(atoms.num_atoms) + "\n"
            line += (
                "Lattice="
                + '"'
                + " ".join(map(str, (atoms.lattice_mat).flatten()))
                + '"'
                + " Properties=species:S:1:pos:R:3:forces:R:3 energy="
                + str(i["energy"])
                # + ' stress="'
                # + " ".join(map(str, np.array(i["stresses"]).flatten()))
                # + '"'
                + " free_energy="
                + str(i["energy"])
                + ' pbc="T T T"'
                + "\n"
            )
            for m, n, p in zip(
                atoms.elements, atoms.cart_coords, i["forces"]
            ):
                line += (
                    str(m)
                    + " "
                    + " ".join(map(str, n))
                    + " "
                    + " ".join(map(str, p))
                    + "\n"
                )
            # print(line)
            f.write(line)
    for i in mlearn:
        if i["jid"] in test_ids:
            # print(i)
            atoms = Atoms.from_dict(i["atoms"])
            line += str(atoms.num_atoms) + "\n"
            line += (
                "Lattice="
                + '"'
                + " ".join(map(str, (atoms.lattice_mat).flatten()))
                + '"'
                + " Properties=species:S:1:pos:R:3:forces:R:3 energy="
                + str(i["energy"])
                # + ' stress="'
                # + " ".join(map(str, np.array(i["stresses"]).flatten()))
                # + '"'
                + " free_energy="
                + str(i["energy"])
                + ' pbc="T T T"'
                + "\n"
            )
            for m, n, p in zip(
                atoms.elements, atoms.cart_coords, i["forces"]
            ):
                line += (
                    str(m)
                    + " "
                    + " ".join(map(str, n))
                    + " "
                    + " ".join(map(str, p))
                    + "\n"
                )
            f.write(line)
            # print(line)
    for i in mlearn:
        if i["jid"] in test_ids:
            # print(i)
            atoms = Atoms.from_dict(i["atoms"])
            line += str(atoms.num_atoms) + "\n"
            line += (
                "Lattice="
                + '"'
                + " ".join(map(str, (atoms.lattice_mat).flatten()))
                + '"'
                + " Properties=species:S:1:pos:R:3:forces:R:3 energy="
                + str(i["energy"])
                # + ' stress="'
                # + " ".join(map(str, np.array(i["stresses"]).flatten()))
                # + '"'
                + " free_energy="
                + str(i["energy"])
                + ' pbc="T T T"'
                + "\n"
            )
            for m, n, p in zip(
                atoms.elements, atoms.cart_coords, i["forces"]
            ):
                line += (
                    str(m)
                    + " "
                    + " ".join(map(str, n))
                    + " "
                    + " ".join(map(str, p))
                    + "\n"
                )
            # print(line)
            f.write(line)
    f.close()
    pprint.pprint(tut)
    cmd = "rm -rf ./results"
    os.system(cmd)

    yaml_f = 'allegro/configs/tutorial_'+element+'.yaml'

    cmd = 'cp allegro/configs/tutorial.yaml allegro/configs/tutorial_'+element+'.yaml'
    os.system(cmd)
    tmp="  "+element+": 0"
    #replaceAll(yaml_f,"Si: 0",tmp)
    tut['chemical_symbol_to_type'] ={element: 0}
    tut['n_train'] = len(train_ids)
    tut['shuffle'] = False
    tut['n_test'] = len(test_ids)
    tut['n_val'] = len(test_ids)
    tut['batch_size'] = 2
    # tut['r_max'] = 6
    # tut['lmax'] = 2
    # tut['num_layers'] = 2
    # tut['env_embed_multiplicity'] = 64
    # tut['two_body_latent_mlp_latent_dimensions']=[128, 256, 512, 1024]
    # tut['latent_mlp_latent_dimension'] = [128] #[1024, 1024, 1024]
    # tut['edge_eng_mlp_latent_dimension'] = 128
    tut['learning_rate'] = 0.001
    tut['train_val_split '] = 'sequential'
    tut['shuffle']=False
    #pprint.pprint('config',tut)
    with open(yaml_f, "w+") as fp:
        yaml.dump(tut,fp)
    cmd = "nequip-train allegro/configs/tutorial_"+element+".yaml  --equivariance-test"
    os.system(cmd)
    print('FINISHED')
    import torch
    from nequip.utils import Config
    from nequip.model import model_from_config
    from nequip.data import AtomicData, ASEDataset

    device = "cpu"
    if torch.cuda.is_available():
        device = torch.device("cuda")
    config = Config.from_file(
        "results/silicon-tutorial/si/config.yaml"
    )

    # config["train_on_keys"]=["forces", "total_energy"]
    # config["model_builders"] = ["EnergyModel", "PerSpeciesRescale", "ForceOutput", "RescaleEnergyEtc"]
    model = model_from_config(config, initialize=False)
    d = torch.load(
        "results/silicon-tutorial/si/best_model.pth",
        map_location=device,
    )
    model.load_state_dict(d)

    df = pd.DataFrame(mlearn)

    def get_allegro_forces(model=[], atoms=[], cutoff=5):
        ase_atoms = atoms.ase_converter()
        a = AtomicData.from_ase(ase_atoms, cutoff)
        data = AtomicData.to_AtomicDataDict(a)
        tm = TypeMapper(
            chemical_symbol_to_type=config["chemical_symbol_to_type"]
        )
        data = tm(data)
        out = model(data)
        pen = (
            out["total_energy"]
            .squeeze()
            .cpu()
            .detach()
            .numpy()
            .tolist()
        )
        num_atoms = atoms.num_atoms
        pf = out["forces"].squeeze().cpu().detach().numpy()
        return pen, pf, 0


    for i in glob.glob("jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/MLFF/*energy*.zip"):

        if "mlearn" in i and element in i:
            fname_e = (
                "AI-MLFF-energy-"
                + i.split("/")[-1].split("_energy.json.zip")[0]
                + "-test-mae.csv"
            )
            fname_f = (
                "AI-MLFF-forces-"
                + i.split("/")[-1].split("_energy.json.zip")[0]
                + "-test-multimae.csv"
            )
            fname_s = (
                "AI-MLFF-stresses-"
                + i.split("/")[-1].split("_energy.json.zip")[0]
                + "-test-multimae.csv"
            )
            f_e = open(fname_e, "w")
            f_f = open(fname_f, "w")
            # f_s = open(fname_s, "w")

            f_e.write("id,target,prediction\n")
            f_f.write("id,target,prediction\n")
            # f_s.write("id,prediction\n")
            #
            print(i)
            dat = json.loads(
                zipfile.ZipFile(i).read(
                    i.split("/")[-1].split(".zip")[0]
                )
            )
            print(dat["test"])
            for key, val in dat["test"].items():
                entry = df[df["jid"] == key]
                atoms = Atoms.from_dict(entry.atoms.values[0])
                # print(key,val,df[df['jid']==key],atoms)
                # energy,forces=get_alignn_forces(atoms)
                energy, forces, stress = get_allegro_forces(
                    model=model, atoms=atoms
                )
                print(key, val, energy, atoms.num_atoms)
                line = (
                    key
                    + ","
                    + str(entry.energy.values[0])
                    + ","
                    + str(energy)
                    + "\n"
                )
                f_e.write(line)
                line = (
                    key
                    + ","
                    + str(
                        ";".join(
                            map(
                                str,
                                np.array(
                                    entry.forces.values[0]
                                ).flatten(),
                            )
                        )
                    )
                    + ","
                    + str(
                        ";".join(map(str, np.array(forces).flatten()))
                    )
                    + "\n"
                )
                f_f.write(line)
                # line = (
                #     key
                #     + ","
                #     + str(";".join(map(str, np.array(stress).flatten())))
                #     + "\n"
                # )
                # f_s.write(line)
            f_e.close()
            f_f.close()
            # f_s.close()
            zname = fname_e + ".zip"
            with zipfile.ZipFile(zname, "w") as myzip:
                myzip.write(fname_e)

            zname = fname_f + ".zip"
            with zipfile.ZipFile(zname, "w") as myzip:
                myzip.write(fname_f)

            # zname = fname_s + ".zip"
            # with zipfile.ZipFile(zname, "w") as myzip:
            #     myzip.write(fname_s)


In [ ]:
!ls -altr

In [ ]:
yaml_f

In [ ]:
import yaml,pprint
with open(yaml_f, 'r') as stream:
    data_loaded = yaml.load(stream, Loader=yaml.Loader)

pprint.pprint(data_loaded)

In [ ]:
# #Test example material
# from jarvis.core.atoms import Atoms
# from jarvis.db.figshare import get_jid_data
# atoms = Atoms.from_dict(get_jid_data(jid='JVASP-1002',dataset='dft_3d')['atoms'])
# ase_atoms = atoms.ase_converter()
# a = ASEDataset.from_atoms_list([ase_atoms,ase_atoms],extra_fixed_fields={"r_max": 5.0})

In [ ]:
en_df = pd.read_csv('AI-MLFF-energy-mlearn_Si-test-mae.csv.zip')
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(en_df['target'],en_df['prediction']))
%matplotlib inline
import matplotlib.pyplot as plt
plt.plot(en_df['target'],en_df['prediction'],'.')
plt.xlabel('DFT energy(eV)')
plt.ylabel('FF energy(eV)')


In [ ]:
f_df = pd.read_csv('AI-MLFF-forces-mlearn_Si-test-multimae.csv.zip')
target = np.concatenate([np.array(i.split(';'),dtype='float') for i in f_df['target'].values])
pred= np.concatenate([np.array(i.split(';'),dtype='float') for i in f_df['prediction'].values])
print(mean_absolute_error(target,pred))
plt.plot(target,pred,'.')
plt.xlabel('DFT forces(eV/A)')
plt.ylabel('FF forces(eV/A)')